# Módulo AI2 · Soluciones

## Setup (imports, Pydantic, system prompt, helper de API)

In [ ]:
import os
import json
import textwrap
from typing import Optional, Literal

# Colab: userdata
try:
    import google.colab  # noqa
    from google.colab import userdata
    os.environ.setdefault("ANTHROPIC_API_KEY", userdata.get("ANTHROPIC_API_KEY") or "")
except ImportError:
    pass

TIENE_CLAVE = bool(os.environ.get("ANTHROPIC_API_KEY"))
print("clave de API presente:", TIENE_CLAVE)
if not TIENE_CLAVE:
    print("Sin clave: las celdas [API] se saltarán y seguirá el flujo con datos simulados.")

from pydantic import BaseModel, Field

class Pedido(BaseModel):
    """Un pedido extraído de un email de cliente."""
    producto: str = Field(description="nombre del producto pedido")
    unidades: int = Field(ge=1, description="número de unidades, >=1")
    urgente: bool = Field(description="marcar solo si el cliente lo pide explícitamente")
    email_resumen: str = Field(description="resumen de una línea del email")

print(Pedido.model_json_schema()["properties"].keys())

SYSTEM = textwrap.dedent("""
Eres el asistente de pedidos de TiendaSur.

Dominio:
- Responde SOLO sobre pedidos, envíos y facturas de TiendaSur.
- Si el usuario pregunta otra cosa, responde exactamente: "Te paso a soporte".

Reglas:
- NUNCA inventes datos de un pedido: usa la herramienta `get_pedido`.
- Si el usuario pide cambiar algo (dirección, fecha), di: "El cambio debe hacerlo en la app".
- Tonos: cordial de 2-4 frases, sin emojis, sin markdown de encabezados.
""")
print(SYSTEM[:280], "…")

import anthropic

MODELO = "claude-opus-5-5"   # modelo vigente (oct. 2026); para volumen alto: "claude-sonnet-5-5" o "claude-haiku-5-5"

def cliente():
    return anthropic.Anthropic()   # lee ANTHROPIC_API_KEY

def texto_de(msg) -> str:
    """Los modelos actuales razonan antes de responder: msg.content puede empezar por un
    bloque 'thinking'. Nunca uses msg.content[0].text: busca el primer bloque de texto."""
    return next((b.text for b in msg.content if b.type == "text"), "")

def preguntar(mensaje: str, system: str = SYSTEM, max_tokens: int = 4000) -> str:
    """Llamada simple: devuelve el texto de la respuesta."""
    if not TIENE_CLAVE:
        return "[SALTADA: sin clave de API. Con clave, aquí verías la respuesta real.]"
    c = cliente()
    msg = c.messages.create(
        model=MODELO, system=system, max_tokens=max_tokens,   # max_tokens incluye el razonamiento: deja margen
        output_config={"effort": "low"},                      # tarea sencilla: menos razonamiento, menos coste
        messages=[{"role": "user", "content": mensaje}],
    )
    return texto_de(msg)

# Datos y esquema de herramienta del notebook (para los ejercicios)
PEDIDOS = {
    1041: {"producto": "Café grano 1kg", "estado": "entregado", "fecha": "2024-05-10"},
    1042: {"producto": "Tazas TiendaSur", "estado": "preparando", "fecha": "2026-02-06"},
    1043: {"producto": "Prensa francesa", "estado": "enviado", "fecha": "2026-02-04"},
}
def get_pedido(pedido_id: int) -> dict:
    return PEDIDOS.get(pedido_id, {"error": f"pedido {pedido_id} no existe"})

TOOL_SCHEMA = [{
    "name": "get_pedido",
    "description": "Devuelve el estado y detalle de un pedido de TiendaSur por su id.",
    "input_schema": {"type": "object",
                     "properties": {"pedido_id": {"type": "integer"}},
                     "required": ["pedido_id"]},
}]


## Ejercicio 1 · Escapar a la frase exacta

Respuesta orientativa: si la regla dice "exactamente" y aún así el modelo parafrasea, convierte la regla en un **bloque aparte** al final del prompt ("Regla crítica:") y añade un ejemplo few-shot de pregunta-fuera-de-dominio → frase exacta. 3 de 3 pruebas suelen alcanzarlo

## Ejercicio 2 · Campo opcional con None

In [ ]:
from typing import Optional

class Pedido2(BaseModel):
    producto: str
    unidades: int = Field(ge=1)
    urgente: bool
    fecha_deseada: Optional[str] = None
    email_resumen: str

print("schema con fecha opcional ✓")

El modelo la deja `None` si el prompt lo indica y el campo es `Optional`. Pydantic valida (o `ValidationError`)

## Ejercicio 3 · Dos herramientas encadenadas

In [ ]:
PEDIDOS_CLIENTE = {
    7: [1041, 1042, 1043],
}

TOOLS2 = TOOL_SCHEMA + [{
    "name": "listar_pedidos",
    "description": "Devuelve los ids de pedidos de un cliente por su id",
    "input_schema": {"type": "object", "properties": {"cliente_id": {"type": "integer"}},
                      "required": ["cliente_id"]},
}]

def listar_pedidos(cliente_id: int):
    return {"pedidos": PEDIDOS_CLIENTE.get(cliente_id, [])}

# flujo esperado con API:
# 1) modelo pide listar_pedidos(7) → [1041, 1042, 1043]
# 2) modelo pide get_pedido(1041) → entregado
# 3) modelo pide get_pedido(1042) → preparando
# 4) modelo responde en lenguaje natural
print("tools declaradas:", [t["name"] for t in TOOLS2])

El modelo encadenar tool calls si el prompt lo permite; con API el bucle de la sección 2.4 cubre esto con el mismo patrón.

## Ejercicio 4 · 0 unidades rompe Pydantic

In [ ]:
try:
    Pedido.model_validate_json('{"producto": "tazas", "unidades": 0, "urgente": false, "email_resumen": "x"}')
except Exception as e:
    print("ValidationError:", str(e)[:160])

Pydantic: `unidades debe ser >= 1`. Si re-llamas con el error en el prompt, el modelo suele responder con JSON correcto, o pedir que el mínimo sea 1 en la regla (aún mejor: la regla la escribiste tú).

## Ejercicio 5 · Presupuesto (sin API)

Tabla orientativa (con precios de Anthropic, ajusta si usas otro proveedor):

| Estrategia | tokens/turno | coste/turno (sonnet 3.5, aprox) |
|---|---|---|
| Sin caché, sin gestión | 12k + histórico | escala con turnos; carísimo a 20 turnos |
| Con caché de prompt | 12k cacheadas (poco coste) + ~500 nuevas | casi plano |
| Resumen + caché | ~2k + cacheadas | plano y sostenible |

A 500 usuarios × 20 turnos: el orden de magnitud cambia de "decenas de dólares" a "céntimos por turno". Calcula con el precio real del proveedor y entrega tu tabla.